# Agent Platform Colab bootstrap

CAGB-0/CAGB-1 only: exact-SHA CPU provenance smoke + sanitized Drive handoff. No model or GPU execution.

In [ ]:
REPOSITORY_SHA = ""  # exact 40-hex merged commit
REQUEST_ID = ""      # e.g. cagb1-20260925-a

import re
assert re.fullmatch(r"[0-9a-f]{40}", REPOSITORY_SHA), "set an exact repository SHA"
assert re.fullmatch(r"[a-z0-9][a-z0-9-]{2,63}", REQUEST_ID), "set a bounded request id"


In [ ]:
from pathlib import Path
import shutil, subprocess

repo = Path("/content/nyang-repo")
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--filter=blob:none", "https://github.com/hanmiyoo10-alt/-.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REPOSITORY_SHA], check=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
package_root = repo / "tools" / "agent-skill-orchestrator"
sys.path.insert(0, str(package_root))
from benchmarks.colab.request import make_request
from benchmarks.colab.bootstrap import run_bootstrap, validate_bundle

request = make_request(REQUEST_ID, REPOSITORY_SHA)
receipt = run_bootstrap(request, repo, Path("/content/drive/MyDrive"))
bundle = Path("/content/drive/MyDrive") / receipt["drive_handoff_relative_path"]
validate_bundle(bundle)
